# ChaoSSM — Phase 6b: mechanism check + confound control

## What this decides

Phase 6 (core) established, with a validated estimator, that the trained chaossm models have a
**negative conditional Lyapunov exponent on real task inputs** even when free to be chaotic
(r up to 4.0). That characterizes the negative result. But the *why* is not yet pinned down. Two
explanations remain, and they lead to different papers:

- **(A) real mechanism -- training suppresses chaos, and the GATE is the suppressor.** The map is
  locally expanding, but the learned gate g mixes in enough of the previous state (the `(1-g)h`
  path) to pull the effective recurrence into contraction. If true, this is a *mechanism* and
  becomes the paper's headline: "end-to-end training drives the recurrence out of the chaotic
  regime via the gate."
- **(B) estimator artifact -- the conditional estimator loops a single captured input sequence
  over W=400 steps.** A looped short drive can itself induce contraction regardless of the map, so
  the negative lambda could be measurement, not model behaviour.

This notebook runs the two experiments that separate (A) from (B), plus an optional training-time
trajectory of lambda. It **reuses models trained inline** (same recipe as Phase 6, so it is
self-contained and needs no checkpoint upload) but does **no hyperparameter search** -- so it is
cheap (well under an hour on a T4 for the default settings).

## Pre-registered decisions

- **M1 (gate-open):** if forcing the gate open (g == 1) makes the conditional lambda_max cross from
  negative to positive on the free chaotic-r arms, the gate is demonstrably the suppressor ->
  earn the strong "gate suppresses chaos" framing. If lambda stays negative even with the gate
  fully open, the suppression is NOT (solely) the gate -- report the weaker, still-valid
  "trained models contract on-task" statement and attribute contraction to the map/parametrization
  rather than the gate.
- **M2 (confound):** if the conditional lambda_max stays negative when the estimator uses a FRESH
  real input sequence per trajectory (no looping), explanation (B) is ruled out and the
  contraction is a real property of the driven model. If lambda flips to >= 0 under fresh drives,
  the Phase 6 conditional numbers were a looping artifact and must be recomputed before use.
- **M3 (optional, TRACK_TRAINING):** conditional lambda measured at checkpoints during training;
  supports "training drives lambda down" if it starts >= 0 and ends < 0.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase6b_mechanism"):
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"./{project}_results/{phase}"); print(f"[info] not colab ({type(e).__name__})")
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase6b_mechanism


## Configuration

In [2]:
SMOKE_TEST=False
TRACK_TRAINING=False   # M3: measure conditional lambda at checkpoints during training (adds cost)

@dataclass
class Config:
    state_dim:int=48; d_model:int=48; vocab:int=16; n_layers:int=2; mlp_mult:int=4
    steps:int=3500; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0
    dual_lr:float=0.05; alpha_max:float=50.0; corridor_target:float=0.15; corridor_half_width:float=0.10
    lyap_window:int=4; lyap_subsample:int=3; lyap_every:int=20
    benettin_warmup:int=100; benettin_steps:int=400; benettin_traj:int=16
    r_cap_chaotic:float=4.0; r_cap_periodic:float=3.4
    # focus on the FREE chaotic-r arm (the strongest test: it is allowed full chaos)
    arms:tuple=("chaossm_free","chaossm_corridor")
    train_task:str="copy"; seeds:int=5; lr:float=1e-4
    # gate-open sweep values (M1): 1.0 = fully open (pure map path, no state-mixing)
    gate_values:tuple=(1.0,0.9,0.75,0.5)
    track_ckpts:tuple=(0,250,500,1000,2000,3499)

CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.benettin_warmup=10; CFG.benettin_steps=30; CFG.benettin_traj=4
    CFG.gate_values=(1.0,0.5); CFG.track_ckpts=(0,20,39)
    print("*** SMOKE TEST ***")
print(f"TRACK_TRAINING={TRACK_TRAINING}")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))
def arm_is_periodic(a): return a.endswith("_periodic")
def arm_uses_corridor(a): return a.startswith("chaossm_corridor")
def is_lyap_step(cfg,s,w): return s>=w and s%cfg.lyap_every==0

TRACK_TRAINING=False
{
  "state_dim": 48,
  "d_model": 48,
  "vocab": 16,
  "n_layers": 2,
  "mlp_mult": 4,
  "steps": 3500,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "gamma_init": 1.0,
  "dual_lr": 0.05,
  "alpha_max": 50.0,
  "corridor_target": 0.15,
  "corridor_half_width": 0.1,
  "lyap_window": 4,
  "lyap_subsample": 3,
  "lyap_every": 20,
  "benettin_warmup": 100,
  "benettin_steps": 400,
  "benettin_traj": 16,
  "r_cap_chaotic": 4.0,
  "r_cap_periodic": 3.4,
  "arms": [
    "chaossm_free",
    "chaossm_corridor"
  ],
  "train_task": "copy",
  "seeds": 5,
  "lr": 0.0001,
  "gate_values": [
    1.0,
    0.9,
    0.75,
    0.5
  ],
  "track_ckpts": [
    0,
    250,
    500,
    1000,
    2000,
    3499
  ]
}


## 1. Model -- gate-overridable

Identical cell to Phase 4/6, except `step` accepts an optional `gate_override`. When set, the
gate g is replaced by that constant everywhere in the step, so we can measure the recurrence's
Lyapunov exponent as a function of how open the gate is. g=1 => pure map path `phi(Ah)+Bx`
(the (1-g)h state-mixing term vanishes); g<1 mixes in the previous state.

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)
class ChaoticMap(nn.Module):
    def __init__(self,N,r_cap,gi=1.0):
        super().__init__(); self.r_cap=float(r_cap)
        self.w_r=nn.Parameter(torch.zeros(N))
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(gi)))))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
class ChaoSSMCell(nn.Module):
    def __init__(self,cfg,arm):
        super().__init__(); self.N=cfg.state_dim
        r_cap=cfg.r_cap_periodic if arm_is_periodic(arm) else cfg.r_cap_chaotic
        self.phi=ChaoticMap(cfg.state_dim,r_cap,cfg.gamma_init)
        self.A=nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9,1.1)))
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.W_g=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.b_g=nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self,h,x,gate_override=None):
        g=torch.sigmoid(x@self.W_g.T+self.b_g) if gate_override is None \
          else torch.full_like(h,float(gate_override))
        raw=g*self.phi(h@self.A.T)+(1-g)*h+x@self.B.T
        return triangle_fold(raw)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
class Block(nn.Module):
    def __init__(self,cfg,arm):
        super().__init__(); self.arm=arm
        self.ln1=nn.LayerNorm(cfg.d_model); self.cell=ChaoSSMCell(cfg,arm)
        self.ln2=nn.LayerNorm(cfg.d_model); h=cfg.d_model*cfg.mlp_mult
        self.mlp=nn.Sequential(nn.Linear(cfg.d_model,h),nn.GELU(),nn.Linear(h,cfg.d_model))
    def step(self,hs,x):
        u=self.ln1(x); hs=self.cell.step(hs,u); y=x+hs; y=y+self.mlp(self.ln2(y)); return hs,y
    def init_state(self,b,dev,dt): return self.cell.init_state(b,dev,dt)
class ChaoSSMLM(nn.Module):
    def __init__(self,cfg,arm,V,R):
        super().__init__(); self.cfg=cfg; self.arm=arm
        self.emb=nn.Embedding(V,cfg.d_model)
        self.blocks=nn.ModuleList([Block(cfg,arm) for _ in range(cfg.n_layers)])
        self.ln_f=nn.LayerNorm(cfg.d_model); self.readout=nn.Linear(cfg.d_model,R)
    def forward(self,tok,return_traj=False):
        B,T=tok.shape; x=self.emb(tok); dt=x.dtype
        st=[blk.init_state(B,tok.device,dt) for blk in self.blocks]; outs=[]
        trajs=[[] for _ in self.blocks] if return_traj else None
        xseqs=[[] for _ in self.blocks] if return_traj else None
        for t in range(T):
            xt=x[:,t]
            for li,blk in enumerate(self.blocks):
                hi=xt; st[li],xt=blk.step(st[li],xt)
                if return_traj: trajs[li].append(st[li]); xseqs[li].append(blk.ln1(hi))
            outs.append(self.readout(self.ln_f(xt)))
        lg=torch.stack(outs,1)
        if not return_traj: return lg
        return lg,{li:(torch.stack(trajs[li],1),torch.stack(xseqs[li],1)) for li in range(len(self.blocks))}

## 2. Task (copy) + proxy lambda + trainer (same as Phase 6)

In [4]:
V=CFG.vocab; SEP,PAD,QCUE=V,V+1,V+2; VOCAB_SIZE=V+3; READOUT=V+3
def make_copy(cfg,batch,device,gen):
    L=12; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1); return inp,tgt

from torch.func import jacrev
def _step_vec(cell,h,x,g=None): return cell.step(h.unsqueeze(0),x.unsqueeze(0),gate_override=g).squeeze(0)
def lambda_proxy(cell,traj,xseq,W,ns):
    B,T,N=traj.shape
    if T<=W+1: return torch.zeros((),device=traj.device)
    ls=[]
    for b in torch.randperm(B)[:ns]:
        t0=int(torch.randint(0,T-W-1,(1,))); M=torch.eye(N,device=traj.device,dtype=traj.dtype)
        for k in range(W): M=jacrev(lambda z:_step_vec(cell,z,xseq[b,t0+k]))(traj[b,t0+k])@M
        ls.append(torch.log(torch.linalg.matrix_norm(M,ord=2)+1e-12)/W)
    return torch.stack(ls).mean()

def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

## 3. Conditional Benettin estimator, with fresh-vs-looped drive switch (M2)

`fresh_per_traj=False` reproduces Phase 6 (one captured sequence, looped). `fresh_per_traj=True`
captures a NEW real input sequence for every Benettin trajectory, so no looping structure can
accumulate -- the M2 confound control. Also accepts `gate_override` (M1).

In [5]:
@torch.no_grad()
def capture_input_seq(model,cfg,seed,layer=0,length=None):
    length=length or (cfg.benettin_warmup+cfg.benettin_steps)
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(seed)
    inp,_=make_copy(cfg,1,DEVICE,g); x=model.emb(inp); dt=x.dtype
    st=[blk.init_state(1,DEVICE,dt) for blk in model.blocks]; cap=[]; T=inp.shape[1]
    for t in range(T):
        xt=x[:,t]
        for li,blk in enumerate(model.blocks):
            u=blk.ln1(xt)
            if li==layer: cap.append(u.squeeze(0).detach())
            h=blk.cell.step(st[li],u); st[li]=h; y=xt+h; y=y+blk.mlp(blk.ln2(y)); xt=y
    seq=torch.stack(cap,0)
    if seq.shape[0]<length: seq=seq.repeat((length//seq.shape[0])+1,1)
    return seq[:length]

@torch.no_grad()
def conditional_lambda(model,cfg,seed=0,layer=0,gate_override=None,fresh_per_traj=False):
    cell=model.blocks[layer].cell; N=cfg.state_dim
    total=cfg.benettin_warmup+cfg.benettin_steps
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(2000+seed)
    base_seq=None if fresh_per_traj else capture_input_seq(model,cfg,seed=500+seed,layer=layer,length=total)
    lam_maxes=[]
    for tr in range(cfg.benettin_traj):
        seq=capture_input_seq(model,cfg,seed=9000+seed*97+tr,layer=layer,length=total) if fresh_per_traj else base_seq
        h=torch.full((N,),0.5,device=DEVICE)
        Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=DEVICE))[0]; logsum=torch.zeros(N,device=DEVICE)
        for t in range(total):
            x=seq[t%seq.shape[0]]
            J=jacrev(lambda z:_step_vec(cell,z,x,gate_override))(h)
            h=_step_vec(cell,h,x,gate_override)
            Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
            if t>=cfg.benettin_warmup: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
        lam_maxes.append(float(torch.sort(logsum/cfg.benettin_steps,descending=True)[0][0]))
    return float(np.mean(lam_maxes)), float(np.std(lam_maxes))

# estimator validation (logistic, ln2) -- re-run so this notebook stands alone
def logistic_lyap(r=4.0,n=4000,warm=200,seed=0):
    rng=np.random.default_rng(seed); x=rng.uniform(0.1,0.9); s=0.0; c=0
    for t in range(n+warm):
        d=abs(r*(1-2*x)); x=r*x*(1-x)
        if t>=warm: s+=math.log(d+1e-12); c+=1
    return s/max(c,1)
KNOWN=logistic_lyap(); print(f"[validation] logistic r=4 lambda={KNOWN:.4f} vs ln2={math.log(2):.4f} err={abs(KNOWN-math.log(2)):.4f}")

[validation] logistic r=4 lambda=0.6929 vs ln2=0.6931 err=0.0002


## 4. Train the arms (records proxy lambda; optional checkpoint lambda track for M3)

In [6]:
def train_arm(arm,seed,cfg,track=False):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    model=ChaoSSMLM(cfg,arm,VOCAB_SIZE,READOUT).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    use=arm_uses_corridor(arm); ct=cfg.corridor_target
    lo,hi=ct-cfg.corridor_half_width,ct+cfg.corridor_half_width; a_hi=a_lo=0.0
    warm=max(1,int(cfg.warmup_frac*cfg.steps)); diverged=False; track_rows=[]
    for step in range(cfg.steps):
        inp,tgt=make_copy(cfg,cfg.batch,DEVICE,gen)
        need=use and is_lyap_step(cfg,step,warm)
        out=model(inp,return_traj=need); lg,traj=out if need else (out,None)
        Lt=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        Lp=torch.zeros((),device=DEVICE); lam=float("nan")
        if need and traj:
            ls=[]
            for li,(tr,xs) in traj.items():
                lv=lambda_proxy(model.blocks[li].cell,tr,xs,cfg.lyap_window,cfg.lyap_subsample); ls.append(lv)
                Lp=Lp+a_hi*F.relu(lv-hi)+a_lo*F.relu(lo-lv)
            if ls: lam=float(torch.stack(ls).mean().detach())
        opt.zero_grad(set_to_none=True); (Lt+Lp).backward()
        gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip).item(); opt.step(); sched.step()
        if use and math.isfinite(lam):
            a_hi=float(np.clip(a_hi+cfg.dual_lr*(lam-hi),0,cfg.alpha_max)); a_lo=float(np.clip(a_lo+cfg.dual_lr*(lo-lam),0,cfg.alpha_max))
        if track and step in cfg.track_ckpts:
            lm,_=conditional_lambda(model,cfg,seed=seed,gate_override=None,fresh_per_traj=False)
            track_rows.append(dict(arm=arm,seed=seed,step=step,cond_lambda=lm))
        if not (math.isfinite(Lt.item()) and math.isfinite(gn)): diverged=True; break
    return model,diverged,track_rows

## 5. M1 (gate-open) + M2 (fresh-drive) measurements

In [7]:
rows=[]; track_all=[]; t0=time.time()
for arm in CFG.arms:
    for sd in range(CFG.seeds):
        model,div,tr=train_arm(arm,sd,CFG,track=TRACK_TRAINING); track_all+=tr
        if div:
            rows.append(dict(arm=arm,seed=sd,diverged=True)); print(f"{arm} seed={sd} DIVERGED"); continue
        # M2: baseline (looped) vs fresh-per-traj, gate = learned
        lam_loop,_=conditional_lambda(model,CFG,seed=sd,gate_override=None,fresh_per_traj=False)
        lam_fresh,_=conditional_lambda(model,CFG,seed=sd,gate_override=None,fresh_per_traj=True)
        rec=dict(arm=arm,seed=sd,diverged=False,lam_learned_looped=lam_loop,lam_learned_fresh=lam_fresh)
        # M1: gate-open sweep (fresh drive, the honest setting)
        for gv in CFG.gate_values:
            lm,_=conditional_lambda(model,CFG,seed=sd,gate_override=gv,fresh_per_traj=True)
            rec[f"lam_gate_{gv}"]=lm
        rows.append(rec)
        gate_str=" ".join(f"g={gv}:{rec[f'lam_gate_{gv}']:+.3f}" for gv in CFG.gate_values)
        print(f"{arm:18s} seed={sd} learned(loop {lam_loop:+.3f}/fresh {lam_fresh:+.3f}) | {gate_str}")
DYN=pd.DataFrame(rows); savetable(DYN,"phase6b_mechanism.csv")
if track_all: savetable(pd.DataFrame(track_all),"phase6b_training_track.csv")
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")

chaossm_free       seed=0 learned(loop -0.071/fresh -0.086) | g=1.0:+0.292 g=0.9:+0.233 g=0.75:+0.040 g=0.5:-0.032
chaossm_free       seed=1 learned(loop -0.047/fresh -0.069) | g=1.0:+0.402 g=0.9:+0.303 g=0.75:+0.196 g=0.5:-0.187
chaossm_free       seed=2 learned(loop -0.189/fresh -0.110) | g=1.0:+0.411 g=0.9:+0.327 g=0.75:+0.172 g=0.5:-0.038
chaossm_free       seed=3 learned(loop -0.230/fresh -0.193) | g=1.0:+0.325 g=0.9:+0.213 g=0.75:+0.017 g=0.5:-0.100
chaossm_free       seed=4 learned(loop -0.148/fresh -0.135) | g=1.0:+0.428 g=0.9:+0.302 g=0.75:+0.127 g=0.5:-0.124
chaossm_corridor   seed=0 learned(loop -0.052/fresh -0.118) | g=1.0:+0.288 g=0.9:+0.262 g=0.75:+0.055 g=0.5:-0.234
chaossm_corridor   seed=1 learned(loop -0.223/fresh -0.165) | g=1.0:+0.346 g=0.9:+0.232 g=0.75:+0.058 g=0.5:-0.204
chaossm_corridor   seed=2 learned(loop -0.034/fresh +0.008) | g=1.0:+0.478 g=0.9:+0.415 g=0.75:+0.246 g=0.5:-0.070
chaossm_corridor   seed=3 learned(loop -0.236/fresh -0.189) | g=1.0:+0.288 g=0.9

## 6. Analysis

In [8]:
ok=DYN[~DYN.diverged]
summary={}
for arm in CFG.arms:
    s=ok[ok.arm==arm]
    if not len(s): continue
    row=dict(lam_learned_looped=round(s.lam_learned_looped.mean(),4),
             lam_learned_fresh=round(s.lam_learned_fresh.mean(),4))
    for gv in CFG.gate_values: row[f"lam_gate_{gv}"]=round(s[f"lam_gate_{gv}"].mean(),4)
    summary[arm]=row
SUM=pd.DataFrame(summary).T; print(SUM.to_string()); savetable(SUM.reset_index().rename(columns={"index":"arm"}),"phase6b_summary.csv")

# M2 verdict: does fresh drive keep lambda negative?
free_fresh=ok[ok.arm=="chaossm_free"].lam_learned_fresh
m2_still_neg=bool((free_fresh<0).all()) if len(free_fresh) else None
m2_delta=float((ok.lam_learned_fresh-ok.lam_learned_looped).mean())
# M1 verdict: does opening the gate push lambda positive on the free arm?
gate_open_key=f"lam_gate_{max(CFG.gate_values)}"
free_open=ok[ok.arm=="chaossm_free"][gate_open_key]
m1_open_pos=bool((free_open>0).mean()>=0.5) if len(free_open) else None
free_learned=ok[ok.arm=="chaossm_free"].lam_learned_fresh.mean() if len(ok[ok.arm=='chaossm_free']) else float('nan')
free_open_mean=free_open.mean() if len(free_open) else float('nan')
print(f"\nM2 (confound): fresh-drive mean lambda change vs looped = {m2_delta:+.4f}; "
      f"free-arm fresh lambda all negative = {m2_still_neg}")
print(f"M1 (gate): free-arm lambda  learned={free_learned:+.4f}  ->  gate-open(g={max(CFG.gate_values)})={free_open_mean:+.4f}; "
      f"crosses to positive = {m1_open_pos}")

                  lam_learned_looped  lam_learned_fresh  lam_gate_1.0  lam_gate_0.9  lam_gate_0.75  lam_gate_0.5
chaossm_free                 -0.1370            -0.1184        0.3716        0.2756         0.1104       -0.0962
chaossm_corridor             -0.1235            -0.1211        0.3593        0.2860         0.0991       -0.1334
saved /content/drive/My Drive/ChaoSSM/results/phase6b_mechanism/phase6b_summary.csv

M2 (confound): fresh-drive mean lambda change vs looped = +0.0105; free-arm fresh lambda all negative = True
M1 (gate): free-arm lambda  learned=-0.1184  ->  gate-open(g=1.0)=+0.3716; crosses to positive = True


## 7. Figure -- lambda vs gate openness

In [9]:
fig,ax=plt.subplots(figsize=(6.4,4.4))
gvs=list(CFG.gate_values)
for arm in CFG.arms:
    s=ok[ok.arm==arm]
    if not len(s): continue
    ys=[s[f"lam_gate_{gv}"].mean() for gv in gvs]
    es=[s[f"lam_gate_{gv}"].std() for gv in gvs]
    ax.errorbar(gvs,ys,yerr=es,marker="o",capsize=3,label=arm.replace("chaossm_",""))
    ax.scatter([1.02],[s.lam_learned_fresh.mean()],marker="*",s=120,
               color=ax.lines[-1].get_color(),zorder=5)
ax.axhline(0,color="r",ls="--",lw=1,label="chaos threshold (λ=0)")
ax.set(xlabel="gate value g (1.0 = fully open, pure map path)",
       ylabel="conditional $\\lambda_{max}$ (fresh drive)",
       title="Phase 6b: does opening the gate restore chaos?")
ax.invert_xaxis(); ax.legend(fontsize=8); ax.grid(alpha=.3); fig.tight_layout()
savefig(fig,"phase6b_gate_vs_lambda.png"); plt.close(fig)

if track_all:
    T=pd.DataFrame(track_all); fig,ax=plt.subplots(figsize=(6.4,4))
    for arm in CFG.arms:
        s=T[T.arm==arm].groupby("step").cond_lambda.mean()
        if len(s): ax.plot(s.index,s.values,marker="o",label=arm.replace("chaossm_",""))
    ax.axhline(0,color="r",ls="--",lw=1); ax.set(xlabel="training step",ylabel="conditional $\\lambda_{max}$",
        title="M3: conditional λ over training"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); savefig(fig,"phase6b_lambda_over_training.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase6b_mechanism/phase6b_gate_vs_lambda.png


## 8. Verdict

In [10]:
est_ok=abs(KNOWN-math.log(2))<0.05
if m1_open_pos and m2_still_neg:
    decision=("STRONG FRAMING EARNED. M2: contraction survives fresh (non-looped) drives, so the "
              "negative conditional lambda is a real property of the driven model, not an estimator "
              "artifact. M1: opening the gate (g->1) pushes the free chaotic-r arm's conditional "
              "lambda above 0 -- the gate is demonstrably the suppressor. The paper can state the "
              "mechanism: end-to-end training keeps the gate closed enough to hold the recurrence in "
              "a contracting regime, suppressing the chaos the map is capable of. This becomes the "
              "headline; the periodic-r ablation (Phase 4) and the on-task contraction (Phase 6) are "
              "the supporting evidence.")
elif m2_still_neg and not m1_open_pos:
    decision=("MECHANISM IS NOT (SOLELY) THE GATE. M2: contraction is real (survives fresh drives). "
              "M1: but even with the gate fully open the free arm's conditional lambda stays <= 0, so "
              "the gate is not the sole suppressor -- the map/parametrization itself contracts under "
              "real drives. Report the valid, slightly weaker mechanism: 'under real task inputs the "
              "learned recurrence is contracting regardless of gate state; the capacity for chaos "
              "(positive autonomous/parametric r) is not expressed on-task.' Still a genuine "
              "dynamical characterization, just not a gate story.")
elif not m2_still_neg:
    decision=("CONFOUND CONFIRMED -- RECOMPUTE. M2: with fresh (non-looped) drives the conditional "
              "lambda is no longer reliably negative, so the Phase 6 conditional numbers were at "
              "least partly a looping artifact. Do NOT use the looped conditional lambda in the paper. "
              "Re-run Phase 6's conditional estimates with fresh_per_traj=True before making any "
              "on-task contraction claim. The Phase 4 accuracy-level negative result is unaffected; "
              "only the dynamical-mechanism claim is on hold.")
else:
    decision=("Inconclusive at these settings; inspect the summary table and consider more seeds / "
              "trajectories.")

verdict=dict(smoke_test=SMOKE_TEST, estimator_ok=est_ok,
             estimator_err=round(abs(KNOWN-math.log(2)),4),
             M1_gate=dict(free_learned=round(float(free_learned),4),
                          free_gate_open=round(float(free_open_mean),4),
                          crosses_positive=m1_open_pos, gate_values=list(CFG.gate_values)),
             M2_confound=dict(mean_fresh_minus_looped=round(m2_delta,4),
                              free_fresh_all_negative=m2_still_neg),
             summary={a:summary[a] for a in summary},
             tracked_training=bool(track_all),
             decision=decision)
savejson(verdict,"phase6b_verdict.json")
print(decision); print(); print(json.dumps(verdict,indent=2,default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase6b_mechanism/phase6b_verdict.json
STRONG FRAMING EARNED. M2: contraction survives fresh (non-looped) drives, so the negative conditional lambda is a real property of the driven model, not an estimator artifact. M1: opening the gate (g->1) pushes the free chaotic-r arm's conditional lambda above 0 -- the gate is demonstrably the suppressor. The paper can state the mechanism: end-to-end training keeps the gate closed enough to hold the recurrence in a contracting regime, suppressing the chaos the map is capable of. This becomes the headline; the periodic-r ablation (Phase 4) and the on-task contraction (Phase 6) are the supporting evidence.

{
  "smoke_test": false,
  "estimator_ok": true,
  "estimator_err": 0.0002,
  "M1_gate": {
    "free_learned": -0.1184,
    "free_gate_open": 0.3716,
    "crosses_positive": true,
    "gate_values": [
      1.0,
      0.9,
      0.75,
      0.5
    ]
  },
  "M2_confound": {
    "mean_fresh_minus_loop